In [0]:
# 1. List all Secret Scopes in this Databricks Workspace
print("--- AVAILABLE SECRET SCOPES ---")
scopes = dbutils.secrets.listScopes()
for scope in scopes:
    print(f"Scope Name: {scope.name}")

# 2. Pick the correct scope name from the output above, replace 'YOUR_SCOPE_NAME' below, 
# and uncomment the next lines to see the exact secret keys inside it.

# print("\n--- SECRETS INSIDE SCOPE ---")
# secrets = dbutils.secrets.list(scope="YOUR_SCOPE_NAME")
# for secret in secrets:
#     print(f"Secret Key: {secret.key}")

In [0]:
# ==============================================================================
# UTILITY: ADLS Gen2 Authentication via Service Principal
# ==============================================================================

# Your exact Azure resources
storage_account_name = "datalakeseniorproj012026"
kv_scope_name = "kv-portfolio12026"

# Fetch ALL credentials dynamically from Databricks Secret Scope
try:
    client_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-id")
    tenant_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-tenant-id")
    client_secret = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-secret")
except Exception as e:
    print(f"Failed to retrieve secrets. Error: {e}")
    raise

# Configure the Spark Session to use OAuth 2.0
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("Spark session successfully configured. Zero hardcoded credentials in scope.")

# ------------------------------------------------------------------------------
# TEST THE CONNECTION
# ------------------------------------------------------------------------------
# Using your exact storage account name to check the 'bronze' container
bronze_uri = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

try:
    files = dbutils.fs.ls(bronze_uri)
    print(f"Connection Successful! Connected to: {bronze_uri}")
except Exception as e:
    print(f"Connection Failed. Error: {e}")

In [0]:
# Test the connection to the Medallion Bronze zone
bronze_uri = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"

try:
    files = dbutils.fs.ls(bronze_uri)
    print(f"Connection Successful! Connected to: {bronze_uri}")
except Exception as e:
    print(f"Connection Failed. Error: {e}")